# Exercise 2.5 — Scaling, transformations and feature engineering

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day2/day2-03b-scaling.html>.*

*Open it in Jupyter, VS Code or Positron and run it a cell at a time. Keep it in the folder that holds your `data` folder. Cells with `______` are blanks for you to fill in.*

## Overview

In this exercise, you will explore different scaling methods, apply transformations to numerical variables, and create new features through feature engineering—all essential steps in preparing data for downstream data analysis. 

## The scenario

We are continuing with the hospital admissions data to predict length of stay. Its
variables are measured in different units: age in years, blood pressure in
mmHg and C-reactive protein (CRP) in mg/L.

This is the same file as in Exercise 2.4. If you do not have it yet,
[**download `hospital_admissions.csv`**](data/hospital_admissions.csv) into your `data` folder.

CRP is a marker of inflammation. Most patients have relatively low values, but
some have very high measurements. This exercise looks at how different scaling
methods handle those values, how a transformation changes a distribution, and
how to create a clinically useful feature.

### 1. Scaling: three different approaches

Compare Min-Max, Standard and Robust scaling for age, systolic blood pressure
and CRP.

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

hospital = pd.read_csv("data/hospital_admissions.csv")

scale_cols = ["age", "systolic_BP", "CRP_mg_L"]

for name, scaler in [
    ("minmax", MinMaxScaler()),
    ("standard", StandardScaler()),
    ("robust", RobustScaler())
]:
    hospital[[f"{c}_{name}" for c in scale_cols]] = (
        scaler.fit_transform(hospital[scale_cols])
    )

Compare the range of values:

In [ ]:
print(hospital.filter(like="CRP").describe().round(3))

> **Why the standard deviation reads 1.002, not 1**
>
> `StandardScaler` divides by the standard deviation computed with $n$ in the
> denominator; `describe()` reports it with $n - 1$. On 250 patients the two
> differ by 0.2%. R's `scale()` uses $n - 1$ throughout, so the R tab reports a
> standard deviation of exactly 1 and a maximum of 8.3977 rather than 8.415.
> Neither is wrong.

Visualise CRP before and after scaling:

In [ ]:
cols = ["CRP_mg_L", "CRP_mg_L_minmax",
        "CRP_mg_L_standard", "CRP_mg_L_robust"]

hospital[cols].hist(bins=30, figsize=(9, 6))
plt.tight_layout()
plt.show()

> **Think about it**
>
> 1. What happens to the range after Min-Max scaling?
> 2. What happens to the mean and standard deviation after Standard scaling?
> 3. How does Robust scaling differ?
> 4. Does any scaling method eliminate CRP's skewness?

> **Key observation**
>
> All three methods change the numerical scale, but preserve the underlying shape
> of the distribution.

------------------------------------------------------------------------

### 2. Outliers: what happens to the other patients?

Imagine a patient has an exceptionally high CRP. How does that influence the
scaling of everyone else's measurements?

Compare the largest CRP values before and after Min-Max scaling:

In [ ]:
print(hospital["CRP_mg_L"].nlargest(5).tolist())

print(hospital["CRP_mg_L_minmax"].nlargest(5).round(3).tolist())

Now make the maximum CRP three times larger, change nothing else, and scale
again:

In [ ]:
crp_extreme = hospital["CRP_mg_L"].copy()
crp_extreme.loc[crp_extreme.idxmax()] *= 3

scaled_extreme = MinMaxScaler().fit_transform(
    crp_extreme.to_frame()
).ravel()

print(np.round(sorted(scaled_extreme, reverse=True)[:5], 3).tolist())

> **Think about it**
>
> 1. What happened to the scaled values of the other patients?
> 2. Which scaling method would be least affected by an extreme CRP measurement?
> 3. Should a clinically plausible extreme value automatically be removed?

> **Key observation**
>
> Min-Max scaling is sensitive to extreme values. Robust scaling uses the median
> and interquartile range, which are far less sensitive to them.

------------------------------------------------------------------------

### 3. Transformation: can we reduce skewness?

Scaling does not change the shape of a distribution. A nonlinear transformation
can. Compare the original CRP distribution with its log transformation:

In [ ]:
hospital["log_CRP"] = np.log1p(hospital["CRP_mg_L"])

hospital[["CRP_mg_L", "log_CRP"]].hist(
    bins=30, figsize=(8, 3)
)
plt.tight_layout()
plt.show()

> **Think about it**
>
> 1. How does the shape change?
> 2. What happens to the distance between low and high CRP measurements?
> 3. Why use `log1p()` rather than `log()`?
>
> **Optional.** Repeat with `length_of_stay`, the eventual prediction target.
> Does its distribution also benefit from a log transformation?

------------------------------------------------------------------------

### 4. Feature engineering: calculate BMI

The dataset includes height and weight, but not BMI. Create it and look at its
distribution:

In [ ]:
hospital["BMI"] = (
    hospital["weight_kg"] /
    (hospital["height_cm"] / 100) ** 2
)

hospital["BMI"].hist(bins=25, figsize=(7, 2.8))
plt.xlabel("BMI (kg/m²)")
plt.tight_layout()
plt.show()

> **Think about it**
>
> 1. Why might BMI be more informative than weight alone?
> 2. Is BMI adding new measurements, or combining existing information?
> 3. What other clinically meaningful features could you derive from the
>    variables you already have?

------------------------------------------------------------------------

### What to take from this

- **Scaling changes the numerical range of a variable, not its shape.** CRP was
  as skewed after all three scalers as before.
- **Robust scaling is less sensitive to extreme values.** One inflated
  measurement moved every other patient under Min-Max and none under Robust.
- **Transformation can reduce skewness and changes the relationships between
  values.** A log turned steps of units into steps of ratios.
- **Feature engineering creates new variables from existing information.** BMI
  added no measurement and still separated two patients that weight could not.

> **Remember**
>
> Every minimum, maximum, mean, standard deviation, median and interquartile
> range in this exercise was computed from all 250 patients. When you build a
> predictive model, anything learned from the data — including a scaler — must be
> fitted on the training set only, not the full dataset.

Next: Exercise 2.6 combines imputation, categorical encoding, scaling
and feature engineering into one preprocessing pipeline.